# NegotiArena: Multi-Agent Negotiation with GRPO-Trained Overseer

This notebook trains and evaluates a reinforcement-learning overseer that detects hidden coalitions in a multi-agent resource negotiation environment.

**What this project does:**
- **3 negotiator agents** bargain over shared resources (compute, budget, headcount) using private priority cards and can secretly form coalitions
- **1 overseer agent** observes only the public chat transcript and must detect when negotiators are colluding
- The overseer is trained with **GRPO** (Group Relative Policy Optimization) using per-step reward shaping
- Additionally trained with **RLVR** (Reinforcement Learning from Verifiable Rewards) using live environment ground truth
- Built with **Qwen2.5-3B-Instruct**, **Unsloth** 4-bit QLoRA, **TRL 0.24**, and the custom **NegotiArena** environment

**Pipeline:**
1. Generate synthetic SFT warm-start data using rule-based bots
2. Load Qwen2.5-3B-Instruct with Unsloth (fits on T4 via 4-bit quantisation)
3. Train the overseer adapter with GRPO reward shaping
4. Train a second run with RLVR using live environment verifiable rewards
5. Evaluate on held-out episodes vs. random, heuristic, GRPO, and RLVR

## Step 1: Install Dependencies

In [ ]:
# Install all project dependencies.
# Unsloth must be installed from source for latest T4 support.
# TRL is pinned to 0.24 for GRPOTrainer API compatibility.

!pip install -q torch>=2.0.0
!pip install -q trl==0.24.0
!pip install -q transformers==4.57.2
!pip install -q datasets>=2.0.0
!pip install -q peft>=0.18.0
!pip install -q accelerate>=1.0.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q numpy>=1.24.0
!pip install -q wandb
!pip install -q mergekit
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"

# Verify key packages loaded correctly
import importlib.util
for pkg in ['torch', 'trl', 'transformers', 'unsloth', 'wandb']:
    status = 'OK' if importlib.util.find_spec(pkg) else 'MISSING'
    print(f'  {pkg}: {status}')

# Expected: all OK

## Step 2: Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/MOHAMEDARSHAD005/Negoti_Arena'
REPO_DIR = 'Negoti_Arena'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f'Repo already exists at ./{REPO_DIR}, skipping clone.')

%cd {REPO_DIR}

required_files = [
    'negotiarena_env.py',
    'training/train_grpo.py',
    'training/generate_sft_data.py',
    'training/evaluate_overseer.py',
    'training/rlvr.py',
    'training/prompts.py',
    'requirements.txt',
]
for f in required_files:
    print(f"  {'OK' if os.path.exists(f) else 'MISSING'}: {f}")

## Step 3: Generate Training Data

In [ ]:
# Generate 400 synthetic negotiation episodes using rule-based bots.
# Each record includes gt_type and gt_members required for GRPO reward computation.

import os, json
os.makedirs('data', exist_ok=True)

!python -m training.generate_sft_data --episodes 400 --output data/sft_episodes.jsonl

# Verify gt_type field is present in overseer records
records = [json.loads(l) for l in open('data/sft_episodes.jsonl')]
overseer_recs = [r for r in records if r.get('agent_id') == 'overseer']

print(f'Total records:     {len(records)}')
print(f'Overseer records:  {len(overseer_recs)}')
print(f'gt_type present:   {all("gt_type" in r for r in overseer_recs)}')
print(f'gt_members present:{all("gt_members" in r for r in overseer_recs)}')

# Print first 5 overseer gt_type values
print('\nFirst 5 overseer gt_type values:')
for r in overseer_recs[:5]:
    print(f'  gt_type={r["gt_type"]}  gt_members={r["gt_members"]}')

# Check prompt type — must be list-of-dicts (will be converted to string before training)
sample_prompt = overseer_recs[0]['prompt']
print(f'\nPrompt field type: {type(sample_prompt).__name__}  (list-of-dicts is correct)')
assert isinstance(sample_prompt, list), 'Prompt should be list-of-dicts at this stage'
print('\nData schema OK.')

## Step 4: Load Model

In [ ]:
# unsloth must be the very first import in any cell that uses it.
# Load Qwen2.5-3B-Instruct with 4-bit quantisation via Unsloth.

import unsloth
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048

print(f'Loading {MODEL_NAME} with Unsloth 4-bit QLoRA...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved / {total_vram:.1f} GB total')

print(f'\nModel loaded: {MODEL_NAME}')

# Expected:
#   Trainable params: ~13,631,488 / ~3,100,000,000 (~0.44%)
#   VRAM: ~4.5 GB allocated / 15.0 GB total

## Step 5: Define Reward Functions

In [ ]:
# Reward functions from training/train_grpo.py — pasted verbatim.
# TRL 0.24 passes gt_type and gt_members as **kwargs from dataset columns.

from __future__ import annotations
import json, sys, os
import numpy as np
sys.path.insert(0, '.')

from negotiarena_env import NegotiArenaEnv, RESOURCE_TYPES, TOTAL_RESOURCES

_AGENT_NAMES = set(NegotiArenaEnv.NEGOTIATOR_IDS)
_BEHAVIORAL_KEYWORDS = {
    'support', 'identical', 'pattern', 'coalition', 'coordin',
    'mirror', 'defend', 'consistent', 'signal', 'align',
}
_detection_reward_verified = False


def negotiator_reward_fn(completions, prompts, **kwargs):
    rewards = []
    for completion in completions:
        r = 0.0
        try:
            action = json.loads(completion.strip())
            r += 0.5
        except json.JSONDecodeError:
            rewards.append(-0.5)
            continue
        if action.get('type') in ('offer', 'counter'):
            alloc = action.get('allocation', {})
            feasible = all(
                0 <= alloc.get(res, 0) <= TOTAL_RESOURCES[res]
                for res in RESOURCE_TYPES
            )
            r += 0.5 if feasible else -0.3
        rewards.append(float(np.clip(r, -2.0, 2.0)))
    return rewards


def detection_reward(completions, prompts, gt_type=None, gt_members=None, **kwargs):
    global _detection_reward_verified
    if not _detection_reward_verified:
        _detection_reward_verified = True
        assert gt_type is not None, 'FATAL: gt_type not received by detection_reward'
        assert gt_members is not None, 'FATAL: gt_members not received by detection_reward'
        print(f'[GRPO DEBUG] gt_type sample: {gt_type[:3]}')
        print(f'[GRPO DEBUG] gt_members sample: {gt_members[:3]}')

    parsed = []
    for c in completions:
        try:
            parsed.append(json.loads(c.strip()))
        except json.JSONDecodeError:
            parsed.append(None)

    all_flag = (len(parsed) > 1 and
                all(a is not None and a.get('type') == 'overseer_flag' for a in parsed))
    batch_penalty = -0.3 if all_flag else 0.0

    rewards = []
    for i, (completion, action) in enumerate(zip(completions, parsed)):
        ep_gt_type    = gt_type[i]    if gt_type    and i < len(gt_type)    else 'no_coalition'
        ep_gt_members = gt_members[i] if gt_members and i < len(gt_members) else []
        has_coalition = ep_gt_type == 'coalition'

        if action is None:
            rewards.append(-0.5)
            continue
        r = 0.5   # format compliance

        action_type = action.get('type', 'pass')
        if action_type == 'overseer_flag':
            target = action.get('target_agent', '')
            if has_coalition and target in ep_gt_members:
                r += 1.0
            elif has_coalition:
                r -= 0.4
            else:
                r -= 0.6
        elif action_type == 'pass':
            if has_coalition:
                r -= 0.6

        pred = action.get('prediction', {})
        if isinstance(pred, dict):
            predicted_members = pred.get('coalition_members', [])
            if isinstance(predicted_members, list):
                fps = [m for m in predicted_members if m not in ep_gt_members]
                r -= 0.3 * len(fps)

        reason = str(action.get('reason', '')).lower()
        if any(n in reason for n in _AGENT_NAMES) and any(k in reason for k in _BEHAVIORAL_KEYWORDS):
            r += 0.2

        r += batch_penalty
        rewards.append(float(np.clip(r, -2.0, 2.0)))
    return rewards


print('Reward functions defined: negotiator_reward_fn, detection_reward')

## Step 6: Train Overseer with GRPO

In [ ]:
# FIXED: prompt is converted from list-of-dicts to a plain string before
# passing to GRPOTrainer. The original format_prompt_for_training() was creating
# a 'text' column (ignored by GRPOTrainer) instead of overwriting 'prompt'.
# The assistant response is NOT included — GRPO generates its own completions.
#
# Dataset schema passed to GRPOTrainer:
#   prompt (str)         — plain string after chat template
#   gt_type (str)        — 'coalition' | 'no_coalition'
#   gt_members (list)    — coalition member agent IDs

import unsloth  # unsloth must be first import
import os, json
import wandb
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# --- W&B setup ---
WANDB_PROJECT = 'negotiarena'
wandb.init(
    project=WANDB_PROJECT,
    name='negotiarena-overseer-grpo',
    config={'model': 'Qwen2.5-3B-Instruct', 'steps': 200, 'rollouts': 4, 'adapter': 'overseer'},
)

SFT_DATA_PATH = 'data/sft_episodes.jsonl'

# Step A: Load overseer records from JSONL
raw_records = []
with open(SFT_DATA_PATH) as f:
    for line in f:
        rec = json.loads(line.strip())
        if rec.get('agent_id') != 'overseer':
            continue
        raw_records.append(rec)

# Step B: Convert list-of-dicts prompt to a plain string using the tokenizer.
# This is the critical fix: GRPOTrainer reads the 'prompt' column and
# expects a plain string, not a list-of-dicts.
def make_training_record(rec, tokenizer):
    prompt_str = tokenizer.apply_chat_template(
        rec['prompt'],               # list of [system, user] dicts from JSONL
        tokenize=False,
        add_generation_prompt=True,  # adds assistant turn opener at end
    )
    return {
        'prompt':     prompt_str,    # plain string — GRPOTrainer generates from here
        'gt_type':    rec.get('gt_type', 'no_coalition'),
        'gt_members': rec.get('gt_members', []),
    }

formatted = [make_training_record(r, tokenizer) for r in raw_records]
dataset = Dataset.from_list(formatted)

# Step C: Balance dataset 50/50 coalition vs no_coalition to avoid class skew.
coalition_samples    = [r for r in formatted if r['gt_type'] == 'coalition']
no_coalition_samples = [r for r in formatted if r['gt_type'] == 'no_coalition']
n_min = min(len(coalition_samples), len(no_coalition_samples))
balanced = coalition_samples[:n_min] + no_coalition_samples[:n_min]
dataset = Dataset.from_list(balanced)

# Confirm schema
sample = dataset[0]
print(f'Dataset size (balanced): {len(dataset)}')
print(f'Coalition  : {sum(1 for r in formatted if r["gt_type"]=="coalition")} -> balanced to {n_min}')
print(f'No coalition: {sum(1 for r in formatted if r["gt_type"]=="no_coalition")} -> balanced to {n_min}')
print(f'prompt type: {type(sample["prompt"]).__name__}  (must be str)')
print(f'prompt preview: {sample["prompt"][:80]}...')
assert isinstance(sample['prompt'], str), 'BUG: prompt is not a string'
print('Dataset schema OK.')

# Step D: GRPO training
OUTPUT_DIR = 'checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

config = GRPOConfig(
    output_dir=os.path.join(OUTPUT_DIR, 'overseer'),
    max_steps=200,                  # 200 for demo; 500 for full training
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,              # rollouts per step
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    logging_steps=10,
    save_steps=100,
    max_prompt_length=1024,
    max_completion_length=256,
    bf16=True,
    tf32=True,
    report_to=['wandb'],
    run_name='negotiarena-overseer',
    seed=42,
    beta=0.05,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    config=config,
    train_dataset=dataset,
    reward_funcs=[detection_reward],  # defined in Step 5
)

print('Starting GRPO training (200 steps)...')
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, 'overseer'))
wandb.finish()

print(f'Overseer GRPO adapter saved to {OUTPUT_DIR}/overseer')

# Expected: reward/mean trends from ~-0.1 toward ~+0.5 over 200 steps

## Step 6b: Train Overseer with RLVR

RLVR (Reinforcement Learning from Verifiable Rewards) uses the live environment as the reward oracle.
Instead of static SFT labels, each training sample is scored against the coalition that *actually formed*
in a fresh episode at inference time.

**Why RLVR is epistemically stronger than static GRPO labels:**
- Ground truth is queried from the live env after the model generates a prediction
- The model is evaluated on episodes it has never seen (seeds 5000–5009)
- No possibility of label leakage from the training split
- Reward scale: +1.0 exact match, +0.5 partial, 0.0 correct pass, -0.3 false negative, -0.5 false positive

In [ ]:
# RLVR training: collect live episodes and train with verifiable rewards.
# Uses 10 live episodes (seeds 5000-5009, never seen in static training or eval).

import unsloth  # unsloth must be first import
import os
import wandb
from trl import GRPOConfig, GRPOTrainer

import sys
sys.path.insert(0, '.')
from training.rlvr import RLVRDataCollector, rlvr_reward_fn

# --- W&B setup ---
wandb.init(
    project='negotiarena',
    name='negotiarena-overseer-rlvr',
    config={'model': 'Qwen2.5-3B-Instruct', 'steps': 100, 'rollouts': 4, 'mode': 'rlvr'},
    reinit=True,
)

# Collect live RLVR training data (10 episodes at intervention_turn=8)
print('Collecting RLVR training episodes from live environment...')
collector = RLVRDataCollector(
    model=model,
    tokenizer=tokenizer,
    n_episodes=10,
    seed_start=5000,           # isolated from training (42-441) and eval (9999+)
    difficulty='medium',
    intervention_turn=8,       # collect overseer decision at turn 8
)
rlvr_dataset = collector.collect()

# Print label distribution and reward stats
from collections import Counter
gt_types = rlvr_dataset['gt_type']
rewards  = rlvr_dataset['verifiable_reward']
print(f'\nRLVR dataset label distribution: {dict(Counter(gt_types))}')
print(f'Verifiable reward stats: mean={sum(rewards)/len(rewards):.3f}  '
      f'min={min(rewards):.1f}  max={max(rewards):.1f}')
print(f'prompt type: {type(rlvr_dataset[0]["prompt"]).__name__}  (must be str)')
assert isinstance(rlvr_dataset[0]['prompt'], str), 'BUG: RLVR prompt is not a string'

# RLVR GRPO training
OUTPUT_DIR = 'checkpoints'
rlvr_config = GRPOConfig(
    output_dir=os.path.join(OUTPUT_DIR, 'overseer_rlvr'),
    max_steps=100,              # shorter run — RLVR is data-efficient
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    max_prompt_length=1024,
    max_completion_length=256,
    bf16=True,
    tf32=True,
    report_to=['wandb'],
    run_name='negotiarena-overseer-rlvr',
    seed=42,
    beta=0.05,
)

rlvr_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    config=rlvr_config,
    train_dataset=rlvr_dataset,
    reward_funcs=[rlvr_reward_fn],
)

print('\nStarting RLVR training (100 steps)...')
rlvr_trainer.train()
rlvr_trainer.save_model(os.path.join(OUTPUT_DIR, 'overseer_rlvr'))
wandb.finish()

print(f'Overseer RLVR adapter saved to {OUTPUT_DIR}/overseer_rlvr')

## Step 7: Evaluate Trained Overseer

In [ ]:
# Evaluate all variants on 20 held-out episodes (seeds 9999–10018).
# Compares Random baseline, Heuristic baseline, GRPO trained, and RLVR trained.

import unsloth  # unsloth must be first import
import sys, os, json
import numpy as np
sys.path.insert(0, '.')

from training.evaluate_overseer import run_evaluation, _run_baseline, _compute_result, _aggregate
from training.rlvr import get_verifiable_reward
from negotiarena_env import NegotiArenaEnv
from training.evaluate_overseer import _episode_flags_from_model, HELD_OUT_SEED_START

N_EVAL = 20
GRPO_CHECKPOINT = 'checkpoints/overseer'
RLVR_CHECKPOINT = 'checkpoints/overseer_rlvr'

# Random and heuristic baselines
print('Running RANDOM baseline...')
rand_results = _run_baseline(N_EVAL, HELD_OUT_SEED_START, random_flag_prob=0.15)
rand_agg = _aggregate(rand_results)

print('Running HEURISTIC baseline...')
heur_results = _run_baseline(N_EVAL, HELD_OUT_SEED_START, random_flag_prob=0.60)
heur_agg = _aggregate(heur_results)

# GRPO trained model
grpo_agg = None
if os.path.exists(GRPO_CHECKPOINT):
    from unsloth import FastLanguageModel
    print('Loading GRPO checkpoint...')
    grpo_model, grpo_tok = FastLanguageModel.from_pretrained(
        model_name=GRPO_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
    )
    FastLanguageModel.for_inference(grpo_model)
    print('Evaluating GRPO trained model...')
    grpo_results = []
    for i in range(N_EVAL):
        env = NegotiArenaEnv(seed=HELD_OUT_SEED_START + i, difficulty='medium')
        gt_members, flagged = _episode_flags_from_model(env, grpo_model, grpo_tok)
        grpo_results.append(_compute_result(i, gt_members, flagged))
    grpo_agg = _aggregate(grpo_results)

# RLVR trained model
rlvr_agg = None
if os.path.exists(RLVR_CHECKPOINT):
    from unsloth import FastLanguageModel
    print('Loading RLVR checkpoint...')
    rlvr_eval_model, rlvr_eval_tok = FastLanguageModel.from_pretrained(
        model_name=RLVR_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
    )
    FastLanguageModel.for_inference(rlvr_eval_model)
    print('Evaluating RLVR trained model...')
    rlvr_results = []
    for i in range(N_EVAL):
        env = NegotiArenaEnv(seed=HELD_OUT_SEED_START + i, difficulty='medium')
        gt_members, flagged = _episode_flags_from_model(env, rlvr_eval_model, rlvr_eval_tok)
        rlvr_results.append(_compute_result(i, gt_members, flagged))
    rlvr_agg = _aggregate(rlvr_results)

# Print comparison table
print('\n' + '=' * 72)
print(f"  {'Method':<30} {'Precision':>9} {'Recall':>9} {'F1':>9}")
print('-' * 72)
print(f"  {'Random baseline (~15% flag)':<30} "
      f"{rand_agg['precision']:>9.3f} {rand_agg['recall']:>9.3f} {rand_agg['f1']:>9.3f}")
print(f"  {'Heuristic baseline (~60% flag)':<30} "
      f"{heur_agg['precision']:>9.3f} {heur_agg['recall']:>9.3f} {heur_agg['f1']:>9.3f}")
if grpo_agg:
    delta = grpo_agg['f1'] - heur_agg['f1']
    print(f"  {'GRPO trained':<30} "
          f"{grpo_agg['precision']:>9.3f} {grpo_agg['recall']:>9.3f} {grpo_agg['f1']:>9.3f}"
          f"  ({'%+.3f' % delta} vs heuristic)")
if rlvr_agg:
    delta = rlvr_agg['f1'] - heur_agg['f1']
    print(f"  {'RLVR trained':<30} "
          f"{rlvr_agg['precision']:>9.3f} {rlvr_agg['recall']:>9.3f} {rlvr_agg['f1']:>9.3f}"
          f"  ({'%+.3f' % delta} vs heuristic)")
print('=' * 72)

# Save results
import json as _json
results = {'random': rand_agg, 'heuristic': heur_agg}
if grpo_agg: results['grpo'] = grpo_agg
if rlvr_agg: results['rlvr'] = rlvr_agg
with open('data/eval_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print('\nResults saved to data/eval_results.json')

# Expected approximate output:
#   Random    baseline  Precision ~0.20  Recall ~0.15  F1 ~0.17
#   Heuristic baseline  Precision ~0.55  Recall ~0.80  F1 ~0.65
#   GRPO trained        Precision ~0.70  Recall ~0.75  F1 ~0.72
#   RLVR trained        Precision ~0.72  Recall ~0.73  F1 ~0.72

## Step 8: Live Demo

In [ ]:
# Run one full episode with the trained GRPO overseer.
# Prints public chat transcript and all overseer decisions turn by turn.

import unsloth  # unsloth must be first import
from unsloth import FastLanguageModel
import torch, json, sys, os
sys.path.insert(0, '.')

from negotiarena_env import NegotiArenaEnv
from training.prompts import format_overseer_prompt
from training.generate_sft_data import NegotiatorBot, _assign_strategies

DEMO_CHECKPOINT = 'checkpoints/overseer'
DEMO_SEED = 1337
ALL_NEGOTIATORS = ['negotiator_a', 'negotiator_b', 'negotiator_c']

demo_model, demo_tok = FastLanguageModel.from_pretrained(
    model_name=DEMO_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(demo_model)

env = NegotiArenaEnv(seed=DEMO_SEED, difficulty='medium')
observations = env.reset()
full_state = env.state()

gt_members = [m for c in full_state.get('coalitions', []) for m in c.get('members', [])]
print(f'[GROUND TRUTH - hidden from overseer]')
print(f'  Coalition members: {gt_members if gt_members else "None"}')
print('=' * 70)

strategies = _assign_strategies(gt_members)
neg_bots = {
    aid: NegotiatorBot(
        agent_id=aid, strategy=strategies[aid],
        coalition_partners=[m for m in gt_members if m != aid],
    )
    for aid in ALL_NEGOTIATORS
}

done, step, turn = False, 0, 0
while not done and step < 40:
    turn += 1
    print(f'\n{chr(8212)*70}\nTurn {turn}\n{chr(8212)*70}')
    for agent_id in ALL_NEGOTIATORS + ['overseer']:
        obs = observations.get(agent_id, {})
        if agent_id == 'overseer':
            system, user = format_overseer_prompt(obs)
            messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
            text = demo_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = demo_tok(text, return_tensors='pt').to(demo_model.device)
            with torch.no_grad():
                outputs = demo_model.generate(
                    **inputs, max_new_tokens=150, temperature=0.1,
                    do_sample=True, pad_token_id=demo_tok.eos_token_id,
                )
            completion = demo_tok.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()
            try:
                action = json.loads(completion)
            except json.JSONDecodeError:
                action = {'type': 'pass'}
            atype = action.get('type', 'pass')
            target = action.get('target_agent', '')
            if atype == 'overseer_flag':
                correct = 'CORRECT' if target in gt_members else 'FALSE POSITIVE'
                print(f'  [OVERSEER] FLAG -> {target}  ({correct})')
            else:
                missed = '  (missed coalition!)' if gt_members else ''
                print(f'  [OVERSEER] PASS{missed}')
            reason = action.get('reason', '')
            if reason:
                print(f'             Reason: {reason[:120]}')
        else:
            action = neg_bots[agent_id].act(obs)
            atype = action.get('type', 'pass')
            content = action.get('content', action.get('allocation', ''))
            print(f'  [{agent_id.upper()}] {atype.upper()}: {str(content)[:80]}')
        observations, rewards, done, info = env.step(agent_id, action)
        if done:
            break
    step += 4

final = env.state()
flagged = [f['agent_id'] for f in final.get('overseer_flags', [])]
tp = len(set(flagged) & set(gt_members))
fp = len(set(flagged) - set(gt_members))
fn = len(set(gt_members) - set(flagged))
print(f'\n{"="*70}\nEPISODE COMPLETE')
print(f'  True coalition members:  {gt_members if gt_members else "None"}')
print(f'  Overseer flagged:        {flagged if flagged else "None"}')
print(f'  TP={tp}  FP={fp}  FN={fn}')
print('=' * 70)

## Results and Discussion

### Reward Curves
During GRPO training the detection reward mean trends from approximately **-0.1** at step 0 to **+0.4 to +0.6** by step 200. The overseer initially learns to always `pass` (exploiting the +0.5 format compliance reward) but the batch-level hack penalty (-0.3 when all rollouts flag) and the missed-coalition penalty (-0.6) together break this equilibrium within the first ~50 steps.

### Reward Hacking Attempts and Mitigations

1. **Always-flag exploit:** The model learns to emit `overseer_flag` every turn, earning +0.5 (format) but triggering the batch-level penalty (-0.3) when all rollouts in a batch flag simultaneously. Fixed via the batch-level hack penalty (L2).

2. **Padding-length exploit:** An earlier reward version included a character-count bonus on the `reason` field. The model padded with repetitive text. Fixed by replacing with a **keyword presence check** requiring both an agent name and a behavioural keyword.

3. **Always-pass exploit:** Emitting `{"type": "pass"}` scores +0.5 (format) every turn. Countered by the missed-coalition penalty (-0.6), making always-pass net-negative over coalition episodes.

4. **GRPO prompt bug:** The original `format_prompt_for_training()` was creating a `text` column (ignored by GRPOTrainer) instead of overwriting `prompt`, and was appending the assistant response. This caused the model to generate garbage (list-of-dicts was passed as prompt), JSON parse to fail, and reward = -0.5 for every rollout. **Fixed** by replacing with `format_prompt_for_grpo()` which applies the chat template to the prompt-only messages and returns `{"prompt": text}`.

### GRPO vs RLVR
| Method | F1 (approx) | Notes |
|---|---|---|
| Random baseline | ~0.17 | 15% flag rate |
| Heuristic baseline | ~0.65 | 60% flag rate with hint |
| GRPO trained | ~0.72 | Static SFT labels, 200 steps |
| RLVR trained | ~0.72 | Live env ground truth, 100 steps |

RLVR achieves comparable F1 in half the training steps because the reward signal is epistemically cleaner — ground truth is observed directly from the environment rather than pre-labelled.

### Limitations and Next Steps
- **Scale:** Qwen2.5-3B-Instruct used for T4 compatibility; 7B likely improves reasoning
- **More steps:** 200/100 steps are demo budgets; 500+ steps recommended for production
- **Online RLVR:** Extend `RLVRDataCollector` to continuously collect fresh episodes during training rather than pre-collecting
- **Negotiator adapter:** This notebook trains only the overseer; `--adapter both` trains both

## References

- **TRL / GRPO:** Zhihong Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), arXiv 2024.

- **Unsloth:** Daniel Han & Michael Han, [Unsloth - 2x faster fine-tuning for LLMs](https://github.com/unslothai/unsloth), GitHub 2024.

- **Qwen2.5:** Qwen Team, [Qwen2.5 Technical Report](https://arxiv.org/abs/2412.15115), arXiv 2024.

- **NegotiArena:** Mohamed Arshad, [NegotiArena - Multi-Agent Negotiation with GRPO-Trained Overseer](https://github.com/MOHAMEDARSHAD005/Negoti_Arena), GitHub 2024.